# Challenge 4: Programming an Agent Workflow
A sequential workflow with Search, Critique, and Refine agents that answers questions
by finding data, verifying it, and refining the response before returning it.

In [ ]:
!pip install google-adk requests

## Setup imports and environment

In [ ]:
import os
import logging
import time
from typing import Optional
from google.adk import Agent
from google.adk.agents import SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.adk.tools import google_search

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-fab96dd167ae"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("google_genai.types").setLevel(logging.ERROR)

## Greeter Agent
Receives the user's question and passes it along to the workflow.

In [ ]:
greeter = Agent(
    name="greeter",
    model="gemini-2.0-flash-lite",
    description="Greets the user and acknowledges their question.",
    instruction="""You are the first agent in a research workflow.
Acknowledge the user's question and restate it clearly so the next agent knows exactly what to research.
Do NOT answer the question yourself. Just clarify what needs to be researched.""",
)

## Search Agent
Finds data to answer the question using Google Search.

In [ ]:
search_agent = Agent(
    name="search_agent",
    model="gemini-2.0-flash-lite",
    description="Researches and answers questions using Google Search.",
    instruction="""You are a research specialist. Use Google Search to find accurate, up-to-date information
to answer the question from the previous agent. Provide a detailed answer with key facts and sources.""",
    tools=[google_search],
)

## Critique Agent
Reviews the search agent's response and suggests improvements.

In [ ]:
critique_agent = Agent(
    name="critique_agent",
    model="gemini-2.0-flash-lite",
    description="Reviews responses and suggests improvements.",
    instruction="""You are a critical reviewer. Look at the previous agent's response and evaluate it:
1. Is the information accurate and complete?
2. Is anything missing or unclear?
3. Could the response be better organized or more concise?

Provide specific suggestions for how the response can be improved.
Do NOT rewrite the response yourself. Just provide constructive feedback.""",
)

## Refine Agent
Rewrites the response based on the critique agent's feedback.

In [ ]:
refine_agent = Agent(
    name="refine_agent",
    model="gemini-2.0-flash-lite",
    description="Refines and improves responses based on feedback.",
    instruction="""You are a writing specialist. Take the original response from the search agent
and the feedback from the critique agent, and produce a final, polished answer.
The final answer should be:
- Accurate and well-organized
- Clear and easy to understand
- Concise but complete

Write the final answer directly to the user.""",
)

## Sequential Workflow
Chains the agents: Greeter -> Search -> Critique -> Refine

In [ ]:
workflow = SequentialAgent(
    name="research_workflow",
    description="A sequential workflow that researches, critiques, and refines answers.",
    sub_agents=[greeter, search_agent, critique_agent, refine_agent],
)

## Helper to run the workflow and display all events
Shows output from each sub-agent in the sequence.

In [ ]:
async def run_workflow_verbose(agent, user_message: str) -> str:
    """Run the workflow, print events from each sub-agent, and return the final response."""
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=agent.name, user_id="test_user"
    )
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)

    content = types.Content(
        role="user", parts=[types.Part(text=user_message)]
    )
    final_response = ""
    async for event in runner.run_async(
        user_id="test_user", session_id=session.id, new_message=content
    ):
        # Show which agent is producing output
        if event.author and event.content and event.content.parts:
            txt = event.content.parts[0].text
            if txt:
                print(f"\n--- [{event.author}] ---")
                print(txt.strip()[:500])  # Truncate long outputs for readability
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response

## Test: Run questions through the workflow
Each query goes through Greeter -> Search -> Critique -> Refine

In [ ]:
test_queries = [
    "What are the benefits of renewable energy?",
    "Explain how large language models work.",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    result = await run_workflow_verbose(workflow, query)
    print(f"\n*** FINAL REFINED ANSWER ***")
    print(result)
    time.sleep(5)